In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# ==========================================
# 1. DEFINE THE CNN ARCHITECTURE (From Scratch)
# ==========================================
class FaceRecognitionCNN(nn.Module):
    def __init__(self, num_classes):
        super(FaceRecognitionCNN, self).__init__()
        
        # Conv Layer 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2) 
        
        # Conv Layer 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Fully Connected Layers
        self.flatten = nn.Flatten()
        # Image is resized to 112x112. After two pools, it's 28x28. 
        # 64 channels * 28 * 28 = 50176
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# ==========================================
# 2. DATA PREPARATION
# ==========================================
# ⚠️ REPLACE THIS PATH with the path to your public dataset in Kaggle or Replace with path to dataset https://www.kaggle.com/datasets/jessicali9530/lfw-dataset/code
DATA_DIR = "/kaggle/input/datasets/jessicali9530/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled"

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

print("Loading dataset... (This may take a minute)")
dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

num_public_classes = len(dataset.classes)
print(f"Found {num_public_classes} identities for training.")

# ==========================================
# 3. TRAINING SETUP
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = FaceRecognitionCNN(num_classes=num_public_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==========================================
# 4. TRAINING LOOP
# ==========================================
epochs = 8 # 5 epochs is enough for the model to learn basic facial features

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Print progress every 50 batches
        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(dataloader)}], Loss: {loss.item():.4f}")
            
    avg_loss = running_loss / len(dataloader)
    print(f"--- End of Epoch {epoch+1} | Average Loss: {avg_loss:.4f} ---")

# ==========================================
# 5. SAVE THE WEIGHTS
# ==========================================
# This file will appear in your Kaggle output folder.
torch.save(model.state_dict(), "public_trained_cnn.pth")
print("Training complete! Weights saved as public_trained_cnn.pth")

Loading dataset... (This may take a minute)
Found 5749 identities for training.
Using device: cuda
Epoch [1/8], Batch [50/414], Loss: 8.4790
Epoch [1/8], Batch [100/414], Loss: 7.8240
Epoch [1/8], Batch [150/414], Loss: 8.2574
Epoch [1/8], Batch [200/414], Loss: 8.3059
Epoch [1/8], Batch [250/414], Loss: 8.0397
Epoch [1/8], Batch [300/414], Loss: 8.0922
Epoch [1/8], Batch [350/414], Loss: 8.5479
Epoch [1/8], Batch [400/414], Loss: 8.1170
--- End of Epoch 1 | Average Loss: 8.1991 ---
Epoch [2/8], Batch [50/414], Loss: 8.0791
Epoch [2/8], Batch [100/414], Loss: 7.0683
Epoch [2/8], Batch [150/414], Loss: 7.4123
Epoch [2/8], Batch [200/414], Loss: 7.9921
Epoch [2/8], Batch [250/414], Loss: 7.7263
Epoch [2/8], Batch [300/414], Loss: 7.3538
Epoch [2/8], Batch [350/414], Loss: 7.4002
Epoch [2/8], Batch [400/414], Loss: 7.9707
--- End of Epoch 2 | Average Loss: 7.5684 ---
Epoch [3/8], Batch [50/414], Loss: 6.5794
Epoch [3/8], Batch [100/414], Loss: 7.2044
Epoch [3/8], Batch [150/414], Loss: 8.